### First Demo with Nicheformer
Trying nicheformer with demo Lung data from Xenium


In [0]:
### Data Upload
DATA_H5AD = "/Workspace/Users/hi9100@wayne.edu/nicheFormerUsingDatabricksDemo/nicheFormerTryouts/data/cell_feature_matrix.h5"


In [0]:
##necessary files defining

XENIUM_MEAN_NPY = (
   "/Workspace/Users/hi9100@wayne.edu/nicheFormerUsingDatabricksDemo/nicheFormerTryouts/nicheformer-main/data/model_means/xenium_mean_script.npy"
)
REFERENCE_H5AD = (
    "/Workspace/Users/hi9100@wayne.edu/nicheFormerUsingDatabricksDemo/nicheFormerTryouts/nicheformer-main/data/model_means/model.h5ad"
)

OUT_DIR = "/Workspace/Users/hi9100@wayne.edu/nicheFormerUsingDatabricksDemo/results"


In [0]:
%pip install 'scverse-misc<0.1.5'

In [0]:
%pip install transformers

In [0]:
import os
import importlib.util
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import torch
from transformers import AutoModelForMaskedLM
from huggingface_hub import hf_hub_download

os.makedirs(OUT_DIR, exist_ok=True)

# Note: OUT_DIR creation skipped - on serverless, workspace paths cannot be used for directory creation
# When saving results later, use a Unity Catalog Volume path like: /Volumes/<catalog>/<schema>/<volume>/results


In [0]:
try:
    model = AutoModelForMaskedLM.from_pretrained("aletlvl/Nicheformer", trust_remote_code=True)
    model.eval()

    # AutoTokenizer's auto-resolution fails on this repo (missing class
    # registration) and falls back into a fast/slow-tokenizer conversion
    # error. Bypass it by importing the custom tokenizer class directly.
    module_path = hf_hub_download(repo_id="aletlvl/Nicheformer", filename="tokenization_nicheformer.py")
    spec = importlib.util.spec_from_file_location("tokenization_nicheformer", module_path)
    tok_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(tok_module)
    tokenizer = tok_module.NicheformerTokenizer.from_pretrained("aletlvl/Nicheformer")

    technology_mean = np.load(XENIUM_MEAN_NPY)
    tokenizer._load_technology_mean(technology_mean)

    print("Model and tokenizer loaded.")
except Exception as e:
    print(f"Failed to load model/tokenizer: {e}")
    raise

In [0]:
### "assemble the AnnData objec , Do it only one time and it will save the processed data
import scanpy as sc
import pandas as pd
import os

RAW_MATRIX_H5 = DATA_H5AD  # this currently points at the raw 10x cell_feature_matrix.h5
CELLS_META = "/Workspace/Users/hi9100@wayne.edu/nicheFormerUsingDatabricksDemo/nicheFormerTryouts/data/cells.parquet"

adata = sc.read_10x_h5(RAW_MATRIX_H5)
adata.var_names_make_unique()

cells = pd.read_parquet(CELLS_META).set_index("cell_id")
cells.index = cells.index.str.decode("utf-8")
cells = cells.loc[adata.obs_names]
adata.obsm["spatial"] = cells[["x_centroid", "y_centroid"]].values

sc.pp.subsample(adata, n_obs=3000, random_state=0)

In [0]:
DATA_H5AD = "/Workspace/Users/hi9100@wayne.edu/nicheFormerUsingDatabricksDemo/nicheFormerTryouts/data/xenium_lung_demo.h5ad"
adata.write_h5ad(DATA_H5AD)
print(f"Wrote a real anndata file to {DATA_H5AD}")

In [0]:
DATA_H5AD = "/Workspace/Users/hi9100@wayne.edu/nicheFormerUsingDatabricksDemo/nicheFormerTryouts/data/xenium_lung_demo.h5ad"

In [0]:
###Load data and set metadata columns
adata = ad.read_h5ad(DATA_H5AD)
print(f"Loaded {adata.n_obs} cells x {adata.n_vars} genes")

adata.obs["modality"] = "spatial"
adata.obs["specie"] = "human"
adata.obs["assay"] = "Xenium"

In [0]:
###Extract embeddings

inputs = tokenizer(adata)
print({k: v.shape for k, v in inputs.items()})

 
VOCAB_SIZE = 20345  # from model.nicheformer.embeddings.num_embeddings
inputs["input_ids"] = inputs["input_ids"].clamp(min=0, max=VOCAB_SIZE - 1)

with torch.no_grad():
    embeddings = model.get_embeddings(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        layer=-1,
        with_context=False,
    )
 
embeddings = embeddings.cpu().numpy()
print("Embedding matrix shape:", embeddings.shape)  # expect (n_cells, 512)
 

In [0]:
### Visualize with UMAP

adata.obsm["X_nicheformer"] = embeddings
sc.pp.neighbors(adata, use_rep="X_nicheformer")
sc.tl.umap(adata)

color_by = "cell_type" if "cell_type" in adata.obs.columns else None
sc.pl.umap(adata, color=color_by, show=True)

In [0]:
### Save results
np.save(os.path.join(OUT_DIR, "xenium_lung_nicheformer_embeddings.npy"), embeddings)
adata.write_h5ad(os.path.join(OUT_DIR, "xenium_lung_with_embeddings.h5ad"))

print(f"Saved embeddings and annotated AnnData to {OUT_DIR}")
print("Next: commit and push from the Git folder UI (or `%sh git add . && git commit -m ... && git push`).")